[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/13_tries.ipynb)

*DSA Interview Handbook · Notebook 13 of 14*

# Tries

Prefix trees: share common prefixes so insert, search and 'starts with' cost O(length of the word).

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A trie is a tree of letters where words sharing a beginning share a path, like an autocomplete menu.

**What interviewers really test here:** Prefix queries and combining a trie with DFS.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why tries exist *(Basics)*
- [ ] 2. Building insert, search and startsWith *(Core)*
- [ ] 3. Storing extra data on nodes *(Core)*
- [ ] 4. Tries + DFS *(Core)*
- [ ] 5. Reversed tries and binary tries *(Advanced)*
- [ ] 6. Implicit tries *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **trie** (pronounced "try", from re**trie**val), also called a **prefix tree** or **digital tree**, is a tree-based data structure for storing a set of strings where each edge represents one character and each path from the root spells a prefix. Strings with a common prefix share the nodes of that prefix.



Each node holds a map (or array) of **children** and a boolean **end-of-word** marker. The root represents the empty string.

### Structure and properties

- The root is empty; each child edge is labelled by a character.
- A word is present only if its path exists **and** the last node is marked as end-of-word ("app" may exist as a prefix of "apple" without being a word).
- The depth of a node equals the length of the prefix it represents.
- Maximum children per node = alphabet size σ (26 for lowercase letters, 2 for a binary trie).
- Total nodes ≤ total number of characters inserted + 1.
- An inorder-like DFS visiting children in sorted order lists the words in **lexicographic order**.



#### Child storage choices



| Children as | Lookup | Memory |
| --- | --- | --- |
| fixed array of size σ | O(1) | σ pointers per node (wasteful when sparse) |
| hash map (Python dict) | O(1) average | proportional to actual children |
| sorted list / BST | O(log σ) | compact, ordered |

### Operations and complexity

L = length of the word or prefix, N = number of words.



| Operation | Steps | Time |
| --- | --- | --- |
| Insert | walk/create one node per character, mark the last | O(L) |
| Search (exact) | walk the path, check end marker | O(L) |
| startsWith (prefix) | walk the path only | O(L) |
| Delete | unmark end; remove nodes bottom-up that have no children and aren't ends | O(L) |
| List all words with prefix | walk to the prefix node, DFS below it | O(L + size of output) |



Space: O(total characters × child-pointer cost) in the worst case (no shared prefixes).

### Variants

| Variant | Idea | Use |
| --- | --- | --- |
| **Compressed trie / radix tree (Patricia)** | merge chains of single-child nodes into one edge labelled with a string | IP routing tables, memory-efficient dictionaries |
| **Suffix trie / suffix tree** | a trie of all suffixes of a text | substring search, longest repeated substring |
| **Binary (bitwise) trie** | children are bits 0/1 from the most significant bit | maximum XOR queries, IP prefix matching |
| **Ternary search tree** | each node has less / equal / greater children | space-efficient alternative to tries |
| **Aho–Corasick automaton** | trie of patterns plus failure links | search many patterns in a text at once (antivirus, grep -F) |

### Trie vs hash set vs BST

|   | Trie | Hash set of strings | Balanced BST of strings |
| --- | --- | --- | --- |
| Insert / search | O(L) | O(L) average (hashing) | O(L log N) |
| Prefix query | **O(L)** | O(N · L) | O(L log N) |
| Sorted iteration | yes | no | yes |
| Worst case | no hash collisions | O(N) on collisions | guaranteed |
| Memory | high (many nodes) | low | medium |

### Advantages, disadvantages and applications

**👍 Pros**

##### Advantages

- Prefix queries in O(L), independent of the number of words
- Shared prefixes are stored once
- Words come out in lexicographic order
- No hash collisions



**👎 Cons**

##### Disadvantages

- High memory use: many nodes and child maps
- Slower than a hash set for plain exact lookups in practice
- Pointer-heavy, poor cache locality





#### Applications




- Autocomplete and search suggestions (search engines, IDEs, phone keyboards).
- Spell checkers and word games (Boggle, Scrabble solvers).
- IP routing: longest-prefix match on binary tries.
- T9 predictive text; dictionary compression.
- Multi-pattern text search (Aho–Corasick).

---
# Part 2: Lessons, step by step

## Lesson 1: Why tries exist

*Level: Basics*

Suppose you store 100,000 words and the user types "app". To list words starting with "app", a set or list must check **every** word. Sorting helps a little. A **trie** answers in time proportional to the length of "app" — independent of how many words you stored.



It does this by storing words **letter by letter** in a tree. Words with the same beginning share the same path:



```text
root
 └ a
   └ p
     └ p ◎          "app" ends here
       └ l
         ├ e ◎      "apple"
         └ y ◎      "apply"
```



Each node is just "a dict of children + a flag that says a word ends here".

## Lesson 2: Building insert, search and startsWith

*Level: Core*

```python
class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.end     # path AND end flag

    def startsWith(self, prefix):
        return self._walk(prefix) is not None    # path only
```



Every operation is O(L), L = length of the word. Memory is up to the total number of letters inserted.



**💡 Tip:** In Python, nested dicts are a shorter trie: `node = node.setdefault(ch, {})`, and `node["$"] = True` marks a word end.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def search_vs_prefix():
    root = {}
    for w in ["car", "card", "care", "cat"]:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True
    for q in ["car", "ca", "cap"]:
        node, path = root, []
        for ch in q:
            if ch not in node:
                node = None
                break
            node = node[ch]
            path.append(node)
        found = node is not None and "$" in node
    return root


search_vs_prefix()

{'c': {'a': {'r': {'$': True, 'd': {'$': True}, 'e': {'$': True}},
   't': {'$': True}}}}

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Trie of car, card, care, cat (◎ = word ends)
2. 'car': path exists; search → True, startsWith → True
3. 'ca': path exists; search → False, startsWith → True
4. 'cap': path breaks; search → False, startsWith → False

</details>


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q1](#q1), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 3: Storing extra data on nodes

*Level: Core*

A node can hold more than a flag. Choose what to store from what the question asks:



| Store on each node | Answers |
| --- | --- |
| count of words passing through | "how many words start with p", prefix scores |
| running total of values | map sum pairs |
| the first 3 words (inserted in sorted order) | search suggestions / autocomplete |
| the full word at its end node | word search II, longest word, replace words |
| largest index | prefix and suffix search |



Then a query is a single walk down the prefix, reading what's stored — no further search.

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q9](#q9), [Q7](#q7), [Q5](#q5), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 4: Tries + DFS

*Level: Core*

Because a trie is a tree, you can DFS it, and you can DFS **something else** (a grid, a string) while walking the trie at the same time.




- **Wildcard search** ('.' matches anything): at a '.', try every child.
- **Word Search II**: put all words in a trie, DFS the grid once; each grid step moves one level down the trie, and you stop the moment the current letters aren't a prefix of any word. Much faster than searching each word separately.
- **Concatenated words / word break**: from position i, walk the trie; every end flag is a possible place to cut and recurse.



**💡 Tip:** In Word Search II remove (or mark) a word once found, and delete trie branches that become empty — it prunes later searches.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q13](#q13), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 5: Reversed tries and binary tries

*Level: Advanced*

#### Suffix questions → reverse the words



A trie built from **reversed** words answers "does the stream end with a word?" (stream of characters) and "which words are suffixes of others?" (short encoding).



#### Binary trie for XOR



Insert each number's bits from the highest bit down, so the trie has only 0/1 children. To maximise `x XOR y`, walk the trie for x and at each bit take the **opposite** child if it exists — a 1 in a high bit beats everything below it. O(B) per query, B = number of bits.



With limits (only numbers ≤ m), sort numbers and queries and insert numbers as the limit grows — an **offline** query trick.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def binary_trie_lesson():
    nums = [3, 10, 5]
    B = 4
    root = {}
    for x in nums:
        node = root
        for b in range(B - 1, -1, -1):
            node = node.setdefault((x >> b) & 1, {})
    x, node, cur, path = 5, root, 0, []
    for b in range(B - 1, -1, -1):
        bit = (x >> b) & 1
        if 1 - bit in node:
            cur |= 1 << b
            node = node[1 - bit]
        else:
            node = node[bit]
        path.append(node)
    return cur


binary_trie_lesson()

15

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Insert 3 = 0011 bit by bit (high → low)
2. Insert 10 = 1010 bit by bit (high → low)
3. Insert 5 = 0101 bit by bit (high → low)
4. Bit 3 of 5 is 0: prefer the opposite branch → XOR so far 8
5. Bit 2 of 5 is 1: prefer the opposite branch → XOR so far 12
6. Bit 1 of 5 is 0: prefer the opposite branch → XOR so far 14
7. Bit 0 of 5 is 1: prefer the opposite branch → XOR so far 15

</details>


**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q10](#q10), [Q14](#q14), [Q8](#q8), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 6: Implicit tries

*Level: Advanced*

Sometimes the trie is never built. The numbers 1..n in **lexicographic** (dictionary) order are a preorder walk of a digit trie where the children of x are 10x … 10x + 9.




- **Lexicographical numbers**: walk that tree with arithmetic — go down (×10) if possible, else go to the next sibling (+1), climbing up past 9s.
- **k-th lexicographical number**: count how many numbers ≤ n are in the subtree of a prefix (level by level). If the count fits in k, skip the whole subtree; otherwise go down. O(log² n).



Folder paths are also a trie of parts (`/a/b/c` → a → b → c): remove sub-folders by stopping at the first folder end.

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q20](#q20), [Q12](#q12). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Longest Common Prefix

🟢 Warm-up · *Walk the single-child chain*

Return the longest common prefix of all strings, or `""`.

**Examples**

- `strs = ['flower', 'flow', 'flight']` → `'fl'`
- `strs = ['dog', 'racecar', 'car']` → `''`
- `strs = ['ab', 'a']` → `'a'`

In [ ]:
def longest_common_prefix(strs):
    """
    Return the longest common prefix of all strings, or "".
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_common_prefix, [
    ((['flower', 'flow', 'flight'],), 'fl'),
    ((['dog', 'racecar', 'car'],), ''),
    ((['ab', 'a'],), 'a'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every prefix length of the first word, test it against all words with startswith.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert every word into a trie, then walk from the root while the node has exactly one child and no word ends there.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every prefix length of the first word, test it against all words with startswith.

```python
def longest_common_prefix(strs):
    best = ""
    for k in range(1, len(strs[0]) + 1):
        if all(s.startswith(strs[0][:k]) for s in strs):
            best = strs[0][:k]
    return best
```

**Cost:** O(m² · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert every word into a trie, then walk from the root while the node has exactly one child and no word ends there.

```python
def longest_common_prefix(strs):
    root = {}
    for w in strs:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True
    prefix, node, path = [], root, []
    while len(node) == 1 and "$" not in node:
        ch = next(iter(node))
        prefix.append(ch)
        node = node[ch]
        path.append(node)
    return "".join(prefix)
```

**Cost:** O(total length) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every prefix length of the first word, test it against all words with startswith. That costs O(m² · n) time and O(1) space."

"I can do better: Insert every word into a trie, then walk from the root while the node has exactly one child and no word ends there. That's O(total length) time and O(total length) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/longest-common-prefix)

<a id="q2"></a>
### Q2. Counting Words With a Given Prefix

🟢 Warm-up · *Trie with counts*

Return how many words in `words` start with `pref`.

**Examples**

- `words = ['pay', 'attention', 'practice', 'attend'], pref = 'at'` → `2`
- `words = ['leetcode', 'win', 'loops', 'success'], pref = 'code'` → `0`

In [ ]:
def prefix_count(words, pref):
    """
    Return how many words in words start with pref.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(prefix_count, [
    ((['pay', 'attention', 'practice', 'attend'], 'at'), 2),
    ((['leetcode', 'win', 'loops', 'success'], 'code'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Call startswith on every word.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build a trie with a pass count on every node, then walk the prefix and read the count. For one query this equals the scan; it wins when you answer many prefix queries.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Call startswith on every word.

```python
def prefix_count(words, pref):
    return sum(w.startswith(pref) for w in words)
```

**Cost:** O(n · |pref|) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Build a trie with a pass count on every node, then walk the prefix and read the count. For one query this equals the scan; it wins when you answer many prefix queries.

```python
def prefix_count(words, pref):
    root = {"#": 0}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += 1
    node, path = root, []
    for ch in pref:
        if ch not in node:
            return 0
        node = node[ch]
        path.append(node)
    return node["#"]
```

**Cost:** O(total length) build, O(|pref|) per query time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Call startswith on every word. That costs O(n · |pref|) time and O(1) space."

"I can do better: Build a trie with a pass count on every node, then walk the prefix and read the count. For one query this equals the scan; it wins when you answer many prefix queries. That's O(total length) build, O(|pref|) per query time and O(total length) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/counting-words-with-a-given-prefix)

<a id="q3"></a>
### Q3. Implement Trie (Prefix Tree)

🟡 Interview standard · *Trie node class*

Implement `insert(word)`, `search(word)` (exact match) and `startsWith(prefix)`.

**Examples**

- `Trie() → insert('apple') → search('apple') → search('app') → startsWith('app') → insert('app') → search('app')` → `[None, True, False, True, None, True]`

In [ ]:
class Trie:
    """
    Implement insert(word), search(word) (exact match) and startsWith(prefix).
    """

    def __init__(self):
        pass

    def insert(self, word):
        pass

    def search(self, word):
        pass

    def startsWith(self, prefix):
        pass


# ▶ Run this cell to test your answer
check(Trie, [
    (((), [['insert', ('apple',)], ['search', ('apple',)], ['search', ('app',)], ['startsWith', ('app',)], ['insert', ('app',)], ['search', ('app',)]]), [None, True, False, True, None, True]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list of words; search and startsWith scan every word.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A tree of nodes, one edge per character. Walk or create one node per character; the end flag separates full words from prefixes.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Keep a list of words; search and startsWith scan every word.

```python
class Trie:
    def __init__(self):
        self.words = []

    def insert(self, word):
        self.words.append(word)

    def search(self, word):
        return word in self.words

    def startsWith(self, prefix):
        return any(w.startswith(prefix) for w in self.words)
```

**Cost:** O(n · L) per query time, O(total length) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A tree of nodes, one edge per character. Walk or create one node per character; the end flag separates full words from prefixes.

```python
class Trie:
    class Node:
        def __init__(self):
            self.children, self.end = {}, False

    def __init__(self):
        self.root = self.Node()

    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, self.Node())
        node.end = True

    def _walk(self, s):
        node, path = self.root, []
        for ch in s:
            if ch not in node.children:
                return None, path
            node = node.children[ch]
            path.append(node)
        return node, path

    def search(self, word):
        node, path = self._walk(word)
        return bool(node and node.end)

    def startsWith(self, prefix):
        node, path = self._walk(prefix)
        return node is not None
```

**Cost:** O(L) per operation time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list of words; search and startsWith scan every word. That costs O(n · L) per query time and O(total length) space."

"I can do better: A tree of nodes, one edge per character. Walk or create one node per character; the end flag separates full words from prefixes. That's O(L) per operation time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Trie() → insert('car') → insert('cat') → insert('can') → search('ca') → startsWith('ca') → search('cat')`

1. insert('car')
2. insert('cat')
3. insert('can')
4. search('ca') → path found, end flag False
5. startsWith('ca') → True
6. search('cat') → path found, end flag True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/implement-trie-prefix-tree)

<a id="q4"></a>
### Q4. Design Add and Search Words Data Structure

🟡 Interview standard · *Trie + DFS on '.'*

Support `addWord(word)` and `search(word)`, where '.' in the search matches any single letter.

**Examples**

- `WordDictionary() → addWord('bad') → addWord('dad') → addWord('mad') → search('pad') → search('bad') → search('.ad') → search('b..')` → `[None, None, None, False, True, True, True]`

In [ ]:
class WordDictionary:
    """
    Support addWord(word) and search(word), where '.' in the search matches any
    single letter.
    """

    def __init__(self):
        pass

    def addWord(self, word):
        pass

    def search(self, word):
        pass


# ▶ Run this cell to test your answer
check(WordDictionary, [
    (((), [['addWord', ('bad',)], ['addWord', ('dad',)], ['addWord', ('mad',)], ['search', ('pad',)], ['search', ('bad',)], ['search', ('.ad',)], ['search', ('b..',)]]), [None, None, None, False, True, True, True]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store words in a list; compare every word of the same length letter by letter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store words in a trie. Normal letters follow one edge; '.' tries every child with DFS. Paths that don't exist are dropped immediately.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Store words in a list; compare every word of the same length letter by letter.

```python
class WordDictionary:
    def __init__(self):
        self.words = []

    def addWord(self, word):
        self.words.append(word)

    def search(self, word):
        return any(len(w) == len(word) and all(p in (".", c) for p, c in zip(word, w)) for w in self.words)
```

**Cost:** O(n · L) per search time, O(total length) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Store words in a trie. Normal letters follow one edge; '.' tries every child with DFS. Paths that don't exist are dropped immediately.

```python
class WordDictionary:
    def __init__(self):
        self.root = {}

    def addWord(self, word):
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word):
        hits = []

        def dfs(node, i, path):
            if i == len(word):
                if "$" in node:
                    hits.append(path)
                return "$" in node
            ch = word[i]
            if ch == ".":
                return any(dfs(child, i + 1, path + [child]) for k, child in node.items() if k != "$")
            return ch in node and dfs(node[ch], i + 1, path + [node[ch]])

        found = dfs(self.root, 0, [])
        return found
```

**Cost:** O(L) without dots, O(26^dots · L) worst time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store words in a list; compare every word of the same length letter by letter. That costs O(n · L) per search time and O(total length) space."

"I can do better: Store words in a trie. Normal letters follow one edge; '.' tries every child with DFS. Paths that don't exist are dropped immediately. That's O(L) without dots, O(26^dots · L) worst time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `WordDictionary() → addWord('bad') → addWord('dad') → addWord('mad') → search('pad') → search('bad') → search('.ad') → search('b..')`

1. search('pad') → False
2. search('bad') → True
3. search('.ad') → True
4. search('b..') → True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/design-add-and-search-words-data-structure)

<a id="q5"></a>
### Q5. Replace Words

🟡 Interview standard · *Shortest root in a trie*

Replace each word in `sentence` with the shortest root from `dictionary` that is its prefix.

**Examples**

- `dictionary = ['cat', 'bat', 'rat'], sentence = 'the cattle was rattled by the battery'` → `'the cat was rat by the bat'`
- `dictionary = ['a', 'b', 'c'], sentence = 'aadsfasf absbs bbab cadsfafs'` → `'a a b c'`

In [ ]:
def replace_words(dictionary, sentence):
    """
    Replace each word in sentence with the shortest root from dictionary that is
    its prefix.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(replace_words, [
    ((['cat', 'bat', 'rat'], 'the cattle was rattled by the battery'), 'the cat was rat by the bat'),
    ((['a', 'b', 'c'], 'aadsfasf absbs bbab cadsfafs'), 'a a b c'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every word, test every root with startswith and keep the shortest match.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert roots into a trie. For each word, walk the trie letter by letter and stop at the first end flag — that's the shortest root.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every word, test every root with startswith and keep the shortest match.

```python
def replace_words(dictionary, sentence):
    out = []
    for w in sentence.split():
        roots = [r for r in dictionary if w.startswith(r)]
        out.append(min(roots, key=len) if roots else w)
    return " ".join(out)
```

**Cost:** O(W · R · L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert roots into a trie. For each word, walk the trie letter by letter and stop at the first end flag — that's the shortest root.

```python
def replace_words(dictionary, sentence):
    root = {}
    for r in dictionary:
        node = root
        for ch in r:
            node = node.setdefault(ch, {})
        node["$"] = True
    out = []
    for w in sentence.split():
        node, path, rep = root, [], w
        for i, ch in enumerate(w):
            if ch not in node:
                break
            node = node[ch]
            path.append(node)
            if "$" in node:
                rep = w[:i + 1]
                break
        out.append(rep)
    return " ".join(out)
```

**Cost:** O(total length) time, O(roots' length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every word, test every root with startswith and keep the shortest match. That costs O(W · R · L) time and O(1) space."

"I can do better: Insert roots into a trie. For each word, walk the trie letter by letter and stop at the first end flag — that's the shortest root. That's O(total length) time and O(roots' length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `dictionary = ['cat', 'bat', 'rat'], sentence = 'the cattle was rattled by the battery'`

1. 'the' → 'the'
2. 'cattle' → 'cat'
3. 'was' → 'was'
4. 'rattled' → 'rat'
5. 'by' → 'by'
6. 'the' → 'the'
7. 'battery' → 'bat'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/replace-words)

<a id="q6"></a>
### Q6. Longest Word in Dictionary

🟡 Interview standard · *DFS through end nodes only*

Return the longest word that can be built one letter at a time from other words in the list (ties → smallest lexicographically).

**Examples**

- `words = ['w', 'wo', 'wor', 'worl', 'world']` → `'world'`
- `words = ['a', 'banana', 'app', 'appl', 'ap', 'apply', 'apple']` → `'apple'`

In [ ]:
def longest_word(words):
    """
    Return the longest word that can be built one letter at a time from other
    words in the list (ties → smallest lexicographically).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_word, [
    ((['w', 'wo', 'wor', 'worl', 'world'],), 'world'),
    ((['a', 'banana', 'app', 'appl', 'ap', 'apply', 'apple'],), 'apple'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each word, check that every prefix is in a set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build a trie storing each word at its end node. DFS from the root but only step into children that end a word; the deepest (then smallest) word reached wins.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each word, check that every prefix is in a set.

```python
def longest_word(words):
    s, best = set(words), ""
    for w in words:
        if all(w[:k] in s for k in range(1, len(w))):
            if len(w) > len(best) or (len(w) == len(best) and w < best):
                best = w
    return best
```

**Cost:** O(Σ L²) time, O(total length) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Build a trie storing each word at its end node. DFS from the root but only step into children that end a word; the deepest (then smallest) word reached wins.

```python
def longest_word(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    best = ""

    def dfs(node, path):
        nonlocal best
        for ch in sorted(k for k in node if k != "$"):
            child = node[ch]
            if "$" in child:
                w = child["$"]
                if len(w) > len(best):
                    best = w
                dfs(child, path + [child])

    dfs(root, [])
    return best
```

**Cost:** O(total length) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each word, check that every prefix is in a set. That costs O(Σ L²) time and O(total length) space."

"I can do better: Build a trie storing each word at its end node. DFS from the root but only step into children that end a word; the deepest (then smallest) word reached wins. That's O(total length) time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `words = ['w', 'wo', 'wor', 'worl', 'world']`

1. 'w' can be built letter by letter → new best
2. 'wo' can be built letter by letter → new best
3. 'wor' can be built letter by letter → new best
4. 'worl' can be built letter by letter → new best
5. 'world' can be built letter by letter → new best

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/longest-word-in-dictionary)

<a id="q7"></a>
### Q7. Search Suggestions System

🟡 Interview standard · *Trie nodes keep top 3*

After each typed character of `searchWord`, return up to 3 lexicographically smallest products with that prefix.

**Examples**

- `products = ['mobile', 'mouse', 'moneypot', 'monitor', 'mousepad'], search_word = 'mouse'` → `[['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]`
- `products = ['havana'], search_word = 'tatiana'` → `[[], [], [], [], [], [], []]`

In [ ]:
def suggested_products(products, search_word):
    """
    After each typed character of searchWord, return up to 3 lexicographically
    smallest products with that prefix.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(suggested_products, [
    ((['mobile', 'mouse', 'moneypot', 'monitor', 'mousepad'], 'mouse'), [['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]),
    ((['havana'], 'tatiana'), [[], [], [], [], [], [], []]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each prefix, filter all products and sort them.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort products, then insert them into a trie; each node keeps the first 3 products passing through it (they arrive in sorted order). Each keystroke is one step down the trie.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each prefix, filter all products and sort them.

```python
def suggested_products(products, search_word):
    res = []
    for k in range(1, len(search_word) + 1):
        p = search_word[:k]
        res.append(sorted(x for x in products if x.startswith(p))[:3])
    return res
```

**Cost:** O(L · n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort products, then insert them into a trie; each node keeps the first 3 products passing through it (they arrive in sorted order). Each keystroke is one step down the trie.

```python
def suggested_products(products, search_word):
    root = {}
    for p in sorted(products):
        node = root
        for ch in p:
            node = node.setdefault(ch, {})
            node.setdefault("#", [])
            if len(node["#"]) < 3:
                node["#"].append(p)
    res, node = [], root
    for i, ch in enumerate(search_word):
        node = node.get(ch, {}) if node is not None else {}
        res.append(node.get("#", []))
    return res
```

**Cost:** O(n log n + total length) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each prefix, filter all products and sort them. That costs O(L · n log n) time and O(n) space."

"I can do better: Sort products, then insert them into a trie; each node keeps the first 3 products passing through it (they arrive in sorted order). Each keystroke is one step down the trie. That's O(n log n + total length) time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `products = ['bags', 'baggage', 'banner', 'box', 'cloths'], search_word = 'bags'`

1. Typed 'b' → ['baggage', 'bags', 'banner']
2. Typed 'ba' → ['baggage', 'bags', 'banner']
3. Typed 'bag' → ['baggage', 'bags']
4. Typed 'bags' → ['bags']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/search-suggestions-system)

<a id="q8"></a>
### Q8. Maximum XOR of Two Numbers in an Array

🟡 Interview standard · *Binary trie, greedy opposite bit*

Return the maximum of `nums[i] XOR nums[j]`.

**Examples**

- `nums = [3, 10, 5, 25, 2, 8]` → `28`
- `nums = [14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70]` → `127`

In [ ]:
def find_maximum_xor(nums):
    """
    Return the maximum of nums[i] XOR nums[j].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_maximum_xor, [
    (([3, 10, 5, 25, 2, 8],), 28),
    (([14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70],), 127),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

XOR every pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert each number's bits (high to low) into a binary trie. For a number x, walk down choosing the opposite bit whenever it exists — each such bit adds 2^b to the XOR.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** XOR every pair.

```python
def find_maximum_xor(nums):
    return max(a ^ b for a in nums for b in nums)
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert each number's bits (high to low) into a binary trie. For a number x, walk down choosing the opposite bit whenever it exists — each such bit adds 2^b to the XOR.

```python
def find_maximum_xor(nums):
    B = max(nums).bit_length()
    root = {}
    for x in nums:
        node = root
        for b in range(B - 1, -1, -1):
            node = node.setdefault((x >> b) & 1, {})
    best = 0
    for x in nums:
        node, cur, path = root, 0, []
        for b in range(B - 1, -1, -1):
            bit = (x >> b) & 1
            if 1 - bit in node:
                cur |= 1 << b
                node = node[1 - bit]
            else:
                node = node[bit]
            path.append(node)
        best = max(best, cur)
    return best
```

**Cost:** O(n · B) time, O(n · B) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: XOR every pair. That costs O(n²) time and O(1) space."

"I can do better: Insert each number's bits (high to low) into a binary trie. For a number x, walk down choosing the opposite bit whenever it exists — each such bit adds 2^b to the XOR. That's O(n · B) time and O(n · B) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 10, 5, 25]`

1. Best partner for 3 gives XOR 26
2. Best partner for 10 gives XOR 19
3. Best partner for 5 gives XOR 28
4. Best partner for 25 gives XOR 28

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/maximum-xor-of-two-numbers-in-an-array)

<a id="q9"></a>
### Q9. Map Sum Pairs

🟡 Interview standard · *Trie with prefix totals*

`insert(key, val)` sets a key's value (overwriting). `sum(prefix)` returns the total value of keys starting with prefix.

**Examples**

- `MapSum() → insert('apple', 3) → sum('ap') → insert('app', 2) → sum('ap') → insert('apple', 2) → sum('ap')` → `[None, 3, None, 5, None, 4]`

In [ ]:
class MapSum:
    """
    insert(key, val) sets a key's value (overwriting). sum(prefix) returns the
    total value of keys starting with prefix.
    """

    def __init__(self):
        pass

    def insert(self, key, val):
        pass

    def sum(self, prefix):
        pass


# ▶ Run this cell to test your answer
check(MapSum, [
    (((), [['insert', ('apple', 3)], ['sum', ('ap',)], ['insert', ('app', 2)], ['sum', ('ap',)], ['insert', ('apple', 2)], ['sum', ('ap',)]]), [None, 3, None, 5, None, 4]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a dict; sum scans every key with startswith.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each trie node stores the total of all keys below it. On insert, add the change (new − old value) to every node on the path; sum is one walk to the prefix node.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Keep a dict; sum scans every key with startswith.

```python
class MapSum:
    def __init__(self):
        self.vals = {}

    def insert(self, key, val):
        self.vals[key] = val

    def sum(self, prefix):
        return sum(v for k, v in self.vals.items() if k.startswith(prefix))
```

**Cost:** sum O(n · L) time, O(n · L) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Each trie node stores the total of all keys below it. On insert, add the change (new − old value) to every node on the path; sum is one walk to the prefix node.

```python
class MapSum:
    def __init__(self):
        self.root, self.vals = {"#": 0}, {}

    def insert(self, key, val):
        delta = val - self.vals.get(key, 0)
        self.vals[key] = val
        node = self.root
        for ch in key:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += delta

    def sum(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node:
                return 0
            node = node[ch]
        return node["#"]
```

**Cost:** O(L) per operation time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a dict; sum scans every key with startswith. That costs sum O(n · L) time and O(n · L) space."

"I can do better: Each trie node stores the total of all keys below it. On insert, add the change (new − old value) to every node on the path; sum is one walk to the prefix node. That's O(L) per operation time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `MapSum() → insert('apple', 3) → sum('ap') → insert('app', 2) → sum('ap') → insert('apple', 2) → sum('ap')`

1. insert('apple', 3): add +3 along the path
2. sum('ap') = total stored at the prefix node = 3
3. insert('app', 2): add +2 along the path
4. sum('ap') = total stored at the prefix node = 5
5. insert('apple', 2): add -1 along the path
6. sum('ap') = total stored at the prefix node = 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/map-sum-pairs)

<a id="q10"></a>
### Q10. Short Encoding of Words

🟡 Interview standard · *Trie of reversed words, count leaves*

Encode words as one reference string like 'time#bell#' where each word is a suffix ending at a '#'. Return the shortest possible length.

**Examples**

- `words = ['time', 'me', 'bell']` → `10`
- `words = ['t']` → `2`

In [ ]:
def minimum_length_encoding(words):
    """
    Encode words as one reference string like 'time#bell#' where each word is a
    suffix ending at a '#'. Return the shortest possible length.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(minimum_length_encoding, [
    ((['time', 'me', 'bell'],), 10),
    ((['t'],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Drop every word that is a suffix of another word (check all pairs); sum the rest with +1 each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert reversed words into a trie. Only words ending at a leaf need their own '#'; the answer is Σ (depth + 1) over leaves.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Drop every word that is a suffix of another word (check all pairs); sum the rest with +1 each.

```python
def minimum_length_encoding(words):
    s = set(words)
    keep = [w for w in s if not any(o != w and o.endswith(w) for o in s)]
    return sum(len(w) + 1 for w in keep)
```

**Cost:** O(n² · L) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert reversed words into a trie. Only words ending at a leaf need their own '#'; the answer is Σ (depth + 1) over leaves.

```python
def minimum_length_encoding(words):
    root, ends = {}, {}
    for w in set(words):
        node = root
        for ch in reversed(w):
            node = node.setdefault(ch, {})
        ends[w] = node
    keep = [w for w, node in ends.items() if not node]
    return sum(len(w) + 1 for w in keep)
```

**Cost:** O(total length) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Drop every word that is a suffix of another word (check all pairs); sum the rest with +1 each. That costs O(n² · L) time and O(n) space."

"I can do better: Insert reversed words into a trie. Only words ending at a leaf need their own '#'; the answer is Σ (depth + 1) over leaves. That's O(total length) time and O(total length) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/short-encoding-of-words)

<a id="q11"></a>
### Q11. Lexicographical Numbers

🟡 Interview standard · *Preorder walk of the digit trie*

Return 1..n in lexicographic order using O(n) time and O(1) extra space.

**Examples**

- `n = 13` → `[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]`
- `n = 2` → `[1, 2]`

In [ ]:
def lexical_order(n):
    """
    Return 1..n in lexicographic order using O(n) time and O(1) extra space.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lexical_order, [
    ((13,), [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]),
    ((2,), [1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort the numbers by their string form.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The numbers form an implicit trie (children of x are 10x .. 10x + 9). Walk it in preorder: go to x · 10 if ≤ n; otherwise step to x + 1, climbing up while x ends in 9 or exceeds n.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort the numbers by their string form.

```python
def lexical_order(n):
    return sorted(range(1, n + 1), key=str)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The numbers form an implicit trie (children of x are 10x .. 10x + 9). Walk it in preorder: go to x · 10 if ≤ n; otherwise step to x + 1, climbing up while x ends in 9 or exceeds n.

```python
def lexical_order(n):
    res, x = [], 1
    for _ in range(n):
        res.append(x)
        if x * 10 <= n:
            x *= 10
            move = "go down to x × 10"
        else:
            while x % 10 == 9 or x + 1 > n:
                x //= 10
            x += 1
            move = "next sibling (climbing up if needed)"
    return res
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort the numbers by their string form. That costs O(n log n) time and O(n) space."

"I can do better: The numbers form an implicit trie (children of x are 10x .. 10x + 9). Walk it in preorder: go to x · 10 if ≤ n; otherwise step to x + 1, climbing up while x ends in 9 or exceeds n. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 21`

1. Output 1, then go down to x × 10 → 10
2. Output 10, then next sibling (climbing up if needed) → 11
3. Output 11, then next sibling (climbing up if needed) → 12
4. Output 12, then next sibling (climbing up if needed) → 13
5. Output 13, then next sibling (climbing up if needed) → 14
6. Output 14, then next sibling (climbing up if needed) → 15
7. Output 15, then next sibling (climbing up if needed) → 16
8. Output 16, then next sibling (climbing up if needed) → 17
9. Output 17, then next sibling (climbing up if needed) → 18
10. Output 18, then next sibling (climbing up if needed) → 19
11. Output 19, then next sibling (climbing up if needed) → 2
12. Output 2, then go down to x × 10 → 20
13. Output 20, then next sibling (climbing up if needed) → 21
14. Output 21, then next sibling (climbing up if needed) → 3
15. Output 3, then next sibling (climbing up if needed) → 4

*…and so on.*

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/lexicographical-numbers)

<a id="q12"></a>
### Q12. Remove Sub-Folders from the Filesystem

🟡 Interview standard · *Trie of path parts*

Remove every folder that is inside another folder in the list; return the rest.

**Examples**

- `folder = ['/a', '/a/b', '/c/d', '/c/d/e', '/c/f']` → `['/a', '/c/d', '/c/f']`
- `folder = ['/a', '/a/b/c', '/a/b/d']` → `['/a']`
- `folder = ['/a/b/c', '/a/b/ca', '/a/b/d']` → `['/a/b/c', '/a/b/ca', '/a/b/d']`

In [ ]:
def remove_subfolders(folder):
    """
    Remove every folder that is inside another folder in the list; return the
    rest.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_subfolders, [((["/a", "/a/b", "/c/d", "/c/d/e", "/c/f"],), ["/a", "/c/d", "/c/f"]), ((["/a", "/a/b/c", "/a/b/d"],), ["/a"]), ((["/a/b/c", "/a/b/ca", "/a/b/d"],), ["/a/b/c", "/a/b/ca", "/a/b/d"])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each folder, check whether any other folder + '/' is its prefix.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert each folder's parts ('a', 'b', …) into a trie with an end flag. Then collect folders by DFS, stopping at the first end flag on each path.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each folder, check whether any other folder + '/' is its prefix.

```python
def remove_subfolders(folder):
    return [f for f in folder if not any(g != f and f.startswith(g + "/") for g in folder)]
```

**Cost:** O(n² · L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert each folder's parts ('a', 'b', …) into a trie with an end flag. Then collect folders by DFS, stopping at the first end flag on each path.

```python
def remove_subfolders(folder):
    root = {}
    for f in folder:
        node = root
        for part in f.split("/")[1:]:
            node = node.setdefault(part, {})
        node["$"] = True
    res = []

    def dfs(node, path):
        if "$" in node:
            res.append("/" + "/".join(path))
            return
        for part in node:
            dfs(node[part], path + [part])

    dfs(root, [])
    return res
```

**Cost:** O(total length) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each folder, check whether any other folder + '/' is its prefix. That costs O(n² · L) time and O(1) space."

"I can do better: Insert each folder's parts ('a', 'b', …) into a trie with an end flag. Then collect folders by DFS, stopping at the first end flag on each path. That's O(total length) time and O(total length) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/remove-sub-folders-from-the-filesystem)

<a id="q13"></a>
### Q13. Word Search II

🔴 Stretch · *Trie-guided grid DFS*

Return every word from `words` that can be traced on the board through adjacent cells (each cell used once per word).

**Examples**

- `board = [['o', 'a', 'a', 'n'], ['e', 't', 'a', 'e'], ['i', 'h', 'k', 'r'], ['i', 'f', 'l', 'v']], words = ['oath', 'pea', 'eat', 'rain']` → `['eat', 'oath']`
- `board = [['a', 'b'], ['c', 'd']], words = ['abcb']` → `[]`

In [ ]:
def find_words(board, words):
    """
    Return every word from words that can be traced on the board through
    adjacent cells (each cell used once per word).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_words, [(([["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]], ["oath", "pea", "eat", "rain"]), ["eat", "oath"]),
   (([["a", "b"], ["c", "d"]], ["abcb"]), [])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Run the single-word Word Search DFS separately for every word.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Put all words in a trie. DFS once from each cell, moving down the trie as you go; stop the moment the path is not a prefix of any word. Remove words once found.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Run the single-word Word Search DFS separately for every word.

```python
def find_words(board, words):
    m, n = len(board), len(board[0])

    def exists(word):
        def dfs(r, c, i):
            if i == len(word):
                return True
            if not (0 <= r < m and 0 <= c < n) or board[r][c] != word[i]:
                return False
            ch, board[r][c] = board[r][c], "#"
            ok = any(dfs(r + dr, c + dc, i + 1) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))
            board[r][c] = ch
            return ok
        return any(dfs(r, c, 0) for r in range(m) for c in range(n))
    return [w for w in words if exists(w)]
```

**Cost:** O(W · m·n · 4ᴸ) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Put all words in a trie. DFS once from each cell, moving down the trie as you go; stop the moment the path is not a prefix of any word. Remove words once found.

```python
def find_words(board, words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    m, n = len(board), len(board[0])
    found, path = [], []

    def dfs(r, c, parent):
        ch = board[r][c]
        node = parent.get(ch)
        if node is None:
            return
        path.append((r, c))
        if "$" in node:
            found.append(node.pop("$"))
        board[r][c] = "#"
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and board[nr][nc] in node:
                dfs(nr, nc, node)
        board[r][c] = ch
        path.pop()
        if not node:
            parent.pop(ch)

    for r in range(m):
        for c in range(n):
            if board[r][c] in root:
                dfs(r, c, root)
    return found
```

**Cost:** O(m·n · 4ᴸ) but pruned by the trie time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Run the single-word Word Search DFS separately for every word. That costs O(W · m·n · 4ᴸ) time and O(L) space."

"I can do better: Put all words in a trie. DFS once from each cell, moving down the trie as you go; stop the moment the path is not a prefix of any word. Remove words once found. That's O(m·n · 4ᴸ) but pruned by the trie time and O(total length) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/word-search-ii)

<a id="q14"></a>
### Q14. Prefix and Suffix Search

🔴 Stretch · *Trie of 'suffix#word'*

`WordFilter(words)`; `f(pref, suff)` returns the largest index of a word with that prefix and suffix, or −1.

**Examples**

- `WordFilter(['apple']) → f('a', 'e')` → `[0]`
- `WordFilter(['apple', 'ape', 'apply']) → f('ap', 'e') → f('ap', 'y') → f('b', '')` → `[1, 2, -1]`

In [ ]:
class WordFilter:
    """
    WordFilter(words); f(pref, suff) returns the largest index of a word with
    that prefix and suffix, or −1.
    """

    def __init__(self, words):
        pass

    def f(self, pref, suff):
        pass


# ▶ Run this cell to test your answer
check(WordFilter, [
    (((['apple'],), [['f', ('a', 'e')]]), [0]),
    (((['apple', 'ape', 'apply'],), [['f', ('ap', 'e')], ['f', ('ap', 'y')], ['f', ('b', '')]]), [1, 2, -1]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each query, scan words from the end and check startswith and endswith.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For every word and every suffix s of it, insert s + '#' + word into one trie, storing the word's index on each node (later words overwrite). A query is one walk of suff + '#' + pref.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each query, scan words from the end and check startswith and endswith.

```python
class WordFilter:
    def __init__(self, words):
        self.words = words

    def f(self, pref, suff):
        for i in range(len(self.words) - 1, -1, -1):
            if self.words[i].startswith(pref) and self.words[i].endswith(suff):
                return i
        return -1
```

**Cost:** O(n · L) per query time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** For every word and every suffix s of it, insert s + '#' + word into one trie, storing the word's index on each node (later words overwrite). A query is one walk of suff + '#' + pref.

```python
class WordFilter:
    def __init__(self, words):
        self.root = {}
        for i, w in enumerate(words):
            for k in range(len(w) + 1):
                node = self.root
                for ch in w[k:] + "#" + w:
                    node = node.setdefault(ch, {})
                    node["i"] = i

    def f(self, pref, suff):
        node = self.root
        for ch in suff + "#" + pref:
            if ch not in node:
                return -1
            node = node[ch]
        return node["i"]
```

**Cost:** O(n · L²) build, O(L) query time, O(n · L²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each query, scan words from the end and check startswith and endswith. That costs O(n · L) per query time and O(1) space."

"I can do better: For every word and every suffix s of it, insert s + '#' + word into one trie, storing the word's index on each node (later words overwrite). A query is one walk of suff + '#' + pref. That's O(n · L²) build, O(L) query time and O(n · L²) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/prefix-and-suffix-search)

<a id="q15"></a>
### Q15. Stream of Characters

🔴 Stretch · *Trie of reversed words*

`StreamChecker(words)`; each `query(letter)` returns whether some suffix of the letters seen so far is a word.

**Examples**

- `StreamChecker(['cd', 'f', 'kl']) → query('a') → query('b') → query('c') → query('d') → query('e') → query('f') → query('g') → query('h') → query('i') → query('j') → query('k') → query('l')` → `[False, False, False, True, False, True, False, False, False, False, False, True]`

In [ ]:
class StreamChecker:
    """
    StreamChecker(words); each query(letter) returns whether some suffix of the
    letters seen so far is a word.
    """

    def __init__(self, words):
        pass

    def query(self, letter):
        pass


# ▶ Run this cell to test your answer
check(StreamChecker, [
    (((['cd', 'f', 'kl'],), [['query', ('a',)], ['query', ('b',)], ['query', ('c',)], ['query', ('d',)], ['query', ('e',)], ['query', ('f',)], ['query', ('g',)], ['query', ('h',)], ['query', ('i',)], ['query', ('j',)], ['query', ('k',)], ['query', ('l',)]]), [False, False, False, True, False, True, False, False, False, False, False, True]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep the stream; on each query test every word with endswith.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert words reversed. On each query, walk the trie from the newest letter backwards; reaching an end flag means a word just finished. Only the last maxLen letters matter.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Keep the stream; on each query test every word with endswith.

```python
class StreamChecker:
    def __init__(self, words):
        self.words, self.stream = words, ""

    def query(self, letter):
        self.stream += letter
        return any(self.stream.endswith(w) for w in self.words)
```

**Cost:** O(W · L) per query time, O(stream) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert words reversed. On each query, walk the trie from the newest letter backwards; reaching an end flag means a word just finished. Only the last maxLen letters matter.

```python
class StreamChecker:
    def __init__(self, words):
        from collections import deque
        self.root = {}
        for w in words:
            node = self.root
            for ch in reversed(w):
                node = node.setdefault(ch, {})
            node["$"] = True
        self.recent = deque(maxlen=max(map(len, words)))

    def query(self, letter):
        self.recent.appendleft(letter)
        node, path = self.root, []
        for ch in self.recent:
            if ch not in node:
                break
            node = node[ch]
            path.append(node)
            if "$" in node:
                return True
        return False
```

**Cost:** O(L) per query time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep the stream; on each query test every word with endswith. That costs O(W · L) per query time and O(stream) space."

"I can do better: Insert words reversed. On each query, walk the trie from the newest letter backwards; reaching an end flag means a word just finished. Only the last maxLen letters matter. That's O(L) per query time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `StreamChecker(['cd', 'f', 'kl']) → query('a') → query('c') → query('d') → query('f') → query('k') → query('l')`

1. query('a') → False
2. query('c') → False
3. query('d'): newest letters spell a word backwards → True
4. query('f'): newest letters spell a word backwards → True
5. query('k') → False
6. query('l'): newest letters spell a word backwards → True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/stream-of-characters)

<a id="q16"></a>
### Q16. Concatenated Words

🔴 Stretch · *Trie + word-break DFS*

Return every word that is made of at least two shorter words from the list.

**Examples**

- `words = ['cat', 'cats', 'catsdogcats', 'dog', 'dogcatsdog', 'hippopotamuses', 'rat', 'ratcatdogcat']` → `['catsdogcats', 'dogcatsdog', 'ratcatdogcat']`
- `words = ['cat', 'dog', 'catdog']` → `['catdog']`

In [ ]:
def find_all_concatenated_words(words):
    """
    Return every word that is made of at least two shorter words from the list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_all_concatenated_words, [((["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"],), ["catsdogcats", "dogcatsdog", "ratcatdogcat"]), ((["cat", "dog", "catdog"],), ["catdog"])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each word, try every way to split it into dictionary pieces (no memo).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Put all words in a trie. For a word, walk the trie from position i; at every end flag, recurse from the next position. Memoise positions, and require at least two pieces.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each word, try every way to split it into dictionary pieces (no memo).

```python
def find_all_concatenated_words(words):
    s = set(words)

    def split(w, first):
        for k in range(1, len(w) + 1):
            if w[:k] in s and (k < len(w) or not first) and (k == len(w) or split(w[k:], False)):
                if not (first and k == len(w)):
                    return True
        return False
    return [w for w in words if w and split(w, True)]
```

**Cost:** O(n · 2ᴸ) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Put all words in a trie. For a word, walk the trie from position i; at every end flag, recurse from the next position. Memoise positions, and require at least two pieces.

```python
def find_all_concatenated_words(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True
    res = []
    for w in words:
        memo = {}

        def can(i, pieces):
            if i == len(w):
                return pieces >= 2
            if (i, pieces >= 1) in memo:
                return memo[(i, pieces >= 1)]
            node, ok = root, False
            for j in range(i, len(w)):
                node = node.get(w[j])
                if node is None:
                    break
                if "$" in node and can(j + 1, pieces + 1):
                    ok = True
                    break
            memo[(i, pieces >= 1)] = ok
            return ok

        if w and can(0, 0):
            res.append(w)
    return res
```

**Cost:** O(n · L²) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each word, try every way to split it into dictionary pieces (no memo). That costs O(n · 2ᴸ) time and O(L) space."

"I can do better: Put all words in a trie. For a word, walk the trie from position i; at every end flag, recurse from the next position. Memoise positions, and require at least two pieces. That's O(n · L²) time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `words = ['cat', 'dog', 'cats', 'catsdog', 'dogcat']`

1. 'cat' is not built from shorter words
2. 'dog' is not built from shorter words
3. 'cats' is not built from shorter words
4. 'catsdog' is built from shorter words
5. 'dogcat' is built from shorter words

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/concatenated-words)

<a id="q17"></a>
### Q17. Palindrome Pairs

🔴 Stretch · *Split each word, look up the reverse*

Return all pairs [i, j] (i ≠ j) such that `words[i] + words[j]` is a palindrome.

**Examples**

- `words = ['abcd', 'dcba', 'lls', 's', 'sssll']` → `[[0, 1], [1, 0], [3, 2], [2, 4]]`
- `words = ['bat', 'tab', 'cat']` → `[[0, 1], [1, 0]]`
- `words = ['a', '']` → `[[0, 1], [1, 0]]`

In [ ]:
def palindrome_pairs(words):
    """
    Return all pairs [i, j] (i ≠ j) such that words[i] + words[j] is a
    palindrome.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(palindrome_pairs, [((["abcd", "dcba", "lls", "s", "sssll"],), [[0, 1], [1, 0], [3, 2], [2, 4]]), ((["bat", "tab", "cat"],), [[0, 1], [1, 0]]), ((["a", ""],), [[0, 1], [1, 0]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Concatenate every ordered pair and test it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Split each word w into prefix + suffix at every position. If the prefix is a palindrome, reversed(suffix) placed before w works; if the suffix is a palindrome, reversed(prefix) placed after w works. Look the reverses up in a word → index map (a trie of reversed words gives the same bound).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Concatenate every ordered pair and test it.

```python
def palindrome_pairs(words):
    return [[i, j] for i in range(len(words)) for j in range(len(words)) if i != j and (words[i] + words[j]) == (words[i] + words[j])[::-1]]
```

**Cost:** O(n² · k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Split each word w into prefix + suffix at every position. If the prefix is a palindrome, reversed(suffix) placed before w works; if the suffix is a palindrome, reversed(prefix) placed after w works. Look the reverses up in a word → index map (a trie of reversed words gives the same bound).

```python
def palindrome_pairs(words):
    index = {w: i for i, w in enumerate(words)}
    res = []
    for i, w in enumerate(words):
        for j in range(len(w) + 1):
            pre, suf = w[:j], w[j:]
            if pre == pre[::-1] and suf[::-1] in index and index[suf[::-1]] != i:
                res.append([index[suf[::-1]], i])
            if j != len(w) and suf == suf[::-1] and pre[::-1] in index and index[pre[::-1]] != i:
                res.append([i, index[pre[::-1]]])
    return res
```

**Cost:** O(n · k²) time, O(n · k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Concatenate every ordered pair and test it. That costs O(n² · k) time and O(k) space."

"I can do better: Split each word w into prefix + suffix at every position. If the prefix is a palindrome, reversed(suffix) placed before w works; if the suffix is a palindrome, reversed(prefix) placed after w works. Look the reverses up in a word → index map (a trie of reversed words gives the same bound). That's O(n · k²) time and O(n · k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `words = ['abcd', 'dcba', 'lls', 's', 'sssll']`

1. 'abcd' = '' + 'abcd': '' is a palindrome, so 'dcba' + 'abcd' works
2. 'dcba' = '' + 'dcba': '' is a palindrome, so 'abcd' + 'dcba' works
3. 'lls' = 'll' + 's': 'll' is a palindrome, so 's' + 'lls' works
4. 'sssll' = 'ss' + 'sll': 'ss' is a palindrome, so 'lls' + 'sssll' works

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/palindrome-pairs)

<a id="q18"></a>
### Q18. Maximum XOR With an Element From Array

🔴 Stretch · *Offline queries + binary trie*

For each query [x, m], return the max of x XOR nums[j] over nums[j] ≤ m, or −1 if none.

**Examples**

- `nums = [0, 1, 2, 3, 4], queries = [[3, 1], [1, 3], [5, 6]]` → `[3, 3, 7]`
- `nums = [5, 2, 4, 6, 6, 3], queries = [[12, 4], [8, 1], [6, 3]]` → `[15, -1, 5]`

In [ ]:
def maximize_xor(nums, queries):
    """
    For each query [x, m], return the max of x XOR nums[j] over nums[j] ≤ m, or
    −1 if none.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(maximize_xor, [
    (([0, 1, 2, 3, 4], [[3, 1], [1, 3], [5, 6]]), [3, 3, 7]),
    (([5, 2, 4, 6, 6, 3], [[12, 4], [8, 1], [6, 3]]), [15, -1, 5]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each query, scan every allowed number.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort nums and sort queries by m. Before answering a query, insert every number ≤ m into a binary trie, then walk greedily toward opposite bits.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each query, scan every allowed number.

```python
def maximize_xor(nums, queries):
    res = []
    for x, m in queries:
        ok = [x ^ v for v in nums if v <= m]
        res.append(max(ok) if ok else -1)
    return res
```

**Cost:** O(n · q) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort nums and sort queries by m. Before answering a query, insert every number ≤ m into a binary trie, then walk greedily toward opposite bits.

```python
def maximize_xor(nums, queries):
    nums = sorted(nums)
    B = max(max(nums), max(x for x, _ in queries)).bit_length()
    order = sorted(range(len(queries)), key=lambda q: queries[q][1])
    root, k, res = {}, 0, [-1] * len(queries)
    for qi in order:
        x, m = queries[qi]
        while k < len(nums) and nums[k] <= m:
            node = root
            for b in range(B - 1, -1, -1):
                node = node.setdefault((nums[k] >> b) & 1, {})
            k += 1
        if k:
            node, cur = root, 0
            for b in range(B - 1, -1, -1):
                bit = (x >> b) & 1
                if 1 - bit in node:
                    cur |= 1 << b
                    node = node[1 - bit]
                else:
                    node = node[bit]
            res[qi] = cur
    return res
```

**Cost:** O((n + q) · B + sorting) time, O(n · B) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each query, scan every allowed number. That costs O(n · q) time and O(1) space."

"I can do better: Sort nums and sort queries by m. Before answering a query, insert every number ≤ m into a binary trie, then walk greedily toward opposite bits. That's O((n + q) · B + sorting) time and O(n · B) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [0, 1, 2, 3, 4], queries = [[3, 1], [1, 3], [5, 6]]`

1. Query x = 3, m = 1: trie holds nums ≤ 1 → answer 3
2. Query x = 1, m = 3: trie holds nums ≤ 3 → answer 3
3. Query x = 5, m = 6: trie holds nums ≤ 6 → answer 7

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/maximum-xor-with-an-element-from-array)

<a id="q19"></a>
### Q19. Sum of Prefix Scores of Strings

🔴 Stretch · *Trie with pass counts*

The score of a string is how many words in the list start with it. For each word, return the sum of the scores of all its non-empty prefixes.

**Examples**

- `words = ['abc', 'ab', 'bc', 'b']` → `[5, 4, 3, 2]`
- `words = ['abcd']` → `[4]`

In [ ]:
def sum_prefix_scores(words):
    """
    The score of a string is how many words in the list start with it. For each
    word, return the sum of the scores of all its non-empty prefixes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_prefix_scores, [
    ((['abc', 'ab', 'bc', 'b'],), [5, 4, 3, 2]),
    ((['abcd'],), [4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every prefix of every word, count words starting with it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Insert all words and count how many pass through each node. A word's answer is the sum of the counts along its path.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every prefix of every word, count words starting with it.

```python
def sum_prefix_scores(words):
    return [sum(sum(1 for x in words if x.startswith(w[:k])) for k in range(1, len(w) + 1)) for w in words]
```

**Cost:** O(n² · L²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Insert all words and count how many pass through each node. A word's answer is the sum of the counts along its path.

```python
def sum_prefix_scores(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += 1
    res = []
    for w in words:
        node, total, counts = root, 0, []
        for ch in w:
            node = node[ch]
            total += node["#"]
            counts.append(node["#"])
        res.append(total)
    return res
```

**Cost:** O(total length) time, O(total length) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every prefix of every word, count words starting with it. That costs O(n² · L²) time and O(1) space."

"I can do better: Insert all words and count how many pass through each node. A word's answer is the sum of the counts along its path. That's O(total length) time and O(total length) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `words = ['abc', 'ab', 'bc', 'b']`

1. 'abc': prefixes are shared by [2, 2, 1] words → 5
2. 'ab': prefixes are shared by [2, 2] words → 4
3. 'bc': prefixes are shared by [2, 1] words → 3
4. 'b': prefixes are shared by [2] words → 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/sum-of-prefix-scores-of-strings)

<a id="q20"></a>
### Q20. K-th Smallest in Lexicographical Order

🔴 Stretch · *Count subtree sizes, skip*

Return the k-th smallest number in lexicographic order among 1..n.

**Examples**

- `n = 13, k = 2` → `10`
- `n = 1, k = 1` → `1`
- `n = 100, k = 10` → `17`

In [ ]:
def find_kth_number(n, k):
    """
    Return the k-th smallest number in lexicographic order among 1..n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_kth_number, [
    ((13, 2), 10),
    ((1, 1), 1),
    ((100, 10), 17),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort all numbers as strings and index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Stand on prefix p. Count how many numbers ≤ n start with p (level by level of the digit trie). If that's ≤ the remaining k, skip the whole subtree (p + 1); otherwise go down (p × 10).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort all numbers as strings and index.

```python
def find_kth_number(n, k):
    return int(sorted(map(str, range(1, n + 1)))[k - 1])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Stand on prefix p. Count how many numbers ≤ n start with p (level by level of the digit trie). If that's ≤ the remaining k, skip the whole subtree (p + 1); otherwise go down (p × 10).

```python
def find_kth_number(n, k):
    def count(p):
        total, lo, hi = 0, p, p
        while lo <= n:
            total += min(hi, n) - lo + 1
            lo, hi = lo * 10, hi * 10 + 9
        return total

    cur, k = 1, k - 1
    while k:
        c = count(cur)
        if c <= k:
            k -= c
            cur += 1
        else:
            k -= 1
            cur *= 10
    return cur
```

**Cost:** O(log² n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort all numbers as strings and index. That costs O(n log n) time and O(n) space."

"I can do better: Stand on prefix p. Count how many numbers ≤ n start with p (level by level of the digit trie). If that's ≤ the remaining k, skip the whole subtree (p + 1); otherwise go down (p × 10). That's O(log² n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/tries/k-th-smallest-in-lexicographical-order)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is a trie and why use it over a hash map?</b></summary>

A tree that stores strings character by character with shared prefixes. A hash map answers exact lookups but needs O(N · L) to find all keys with a given prefix; a trie answers prefix queries in O(L).

</details>

<details>
<summary><b>2. What is the time complexity of trie insert and search?</b></summary>

O(L), where L is the length of the word — independent of how many words are stored.

</details>

<details>
<summary><b>3. How do you tell a stored word from a stored prefix?</b></summary>

Each node has an end-of-word flag set on the last node of every inserted word. `search` requires the flag; `startsWith` only requires the path to exist.

</details>

<details>
<summary><b>4. How do you delete a word from a trie?</b></summary>

Unset the end flag on its last node, then walk back up removing nodes that have no children and aren't the end of another word.

</details>

<details>
<summary><b>5. What is a binary trie used for?</b></summary>

Storing numbers bit by bit from the highest bit. To maximise XOR with a query number, greedily follow the opposite bit at each level — O(bits) per query.

</details>

<details>
<summary><b>6. What is a compressed trie (radix tree)?</b></summary>

A trie where chains of nodes with a single child are merged into one edge labelled by a substring. It keeps the same queries with far fewer nodes.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why tries exist
- [ ] I can explain: Building insert, search and startsWith
- [ ] I can explain: Storing extra data on nodes
- [ ] I can explain: Tries + DFS
- [ ] I can explain: Reversed tries and binary tries
- [ ] I can explain: Implicit tries

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Concatenated Words
- [ ] [Q17](#q17) Palindrome Pairs
- [ ] [Q18](#q18) Maximum XOR With an Element From Array
- [ ] [Q19](#q19) Sum of Prefix Scores of Strings
- [ ] [Q20](#q20) K-th Smallest in Lexicographical Order

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.